In [19]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("titanic.csv")

print(dataset)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [20]:
from sklearn.model_selection import LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier

loo = LeaveOneOut()

features = ['Sex', 'Age', 'Pclass', 'Fare']

total_error = 0
total_data = 0

# Lakukan LOO
for iteration, (train_index, test_index) in enumerate(loo.split(dataset), 1):

    # STEP 2: Ambil train dan test
    train_set = dataset.iloc[train_index].copy()
    test_set = dataset.iloc[test_index].copy()

    # STEP 3: Ambil fitur
    train_data = train_set[features].copy()
    test_data = test_set[features].copy()

    # Ubah Sex menjadi angka
    train_data['Sex'] = train_data['Sex'].map({
        'male': 0,
        'female': 1
    })

    test_data['Sex'] = test_data['Sex'].map({
        'male': 0,
        'female': 1
    })

    # Mean Age berdasarkan Pclass dari TRAIN
    age_mean = train_data.groupby('Pclass')['Age'].mean()

    # Isi missing Age pada TRAIN
    train_data['Age'] = train_data['Age'].fillna(
        train_data.groupby('Pclass')['Age'].transform('mean')
    )

    # Isi missing Age pada TEST menggunakan mean TRAIN
    for pclass in age_mean.index:
        test_data.loc[
            (test_data['Pclass'] == pclass) &
            (test_data['Age'].isna()),
            'Age'
        ] = age_mean[pclass]

    # STEP 4: Label
    train_label = train_set['Survived'].copy()
    test_label = test_set['Survived'].copy()

    # STEP 5: Min-Max berdasarkan TRAIN
    train_min = train_data.min()
    train_max = train_data.max()

    train_data = (train_data - train_min) * (1 - 0) / (train_max - train_min) + 0

    # STEP 6: Normalisasi TEST
    test_data = (test_data - train_min) * (1 - 0) / (train_max - train_min) + 0

    # STEP 7: k-NN k=3
    knn = KNeighborsClassifier(n_neighbors=3)

    knn.fit(
        train_data,
        train_label
    )

    prediksi = knn.predict(
        test_data
    )

    # Hitung error
    error = (prediksi != test_label).astype(int)

    total_error += error.sum()
    total_data += len(test_label)

# Error Ratio LOO
error_ratio = total_error / total_data

print("Total error :", total_error)
print("Total data  :", total_data)
print("Error Ratio :", error_ratio)
print("Error (%)   :", error_ratio * 100, "%")


Total error : 153
Total data  : 891
Error Ratio : 0.1717171717171717
Error (%)   : 17.17171717171717 %
